In [2]:
import os

%pwd

'd:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

In [5]:
from src.customer_churn.config.configuration import TrainingPipelineConfig, DataValidationConfig
from src.customer_churn.entity.artifact_entity import DataIngestionArtifacts, DataValidationArtifacts   
from src.customer_churn.exception.exception import CustomerChurnException
from src.customer_churn.logging.logger import log_separator, logging
from src.customer_churn.utils.main_utils.common import read_yaml_file, read_parquet_file
import sys
import pandas as pd
from scipy.stats import ks_2samp


## Create the DataValidation Class:

In [6]:
class DataValidation:
    def __init__(self, data_validation_config: DataValidationConfig, data_ingestion_artifacts: DataIngestionArtifacts):
        try:
            self.data_validation_config = data_validation_config
            self.data_ingestion_artifacts = data_ingestion_artifacts
            self.schema_raw = read_yaml_file(self.data_validation_config.schema_file_path)

            self.schema = {col['name']: col['dtype'] for col in self.schema_raw['columns']}
        except Exception as e:
            raise CustomerChurnException(e, sys)

    def are_column_len_valid(self, df:pd.DataFrame)->bool:
        """
        Takes in the dataframe, checks the number of columns with the expected number of columns defined in the schema (data_schema/schema.yaml)
        """
        try:
            actual_cols = set(df.columns)
            expected_cols = set(self.schema.keys())
            if expected_cols == actual_cols:
                return True
            else:
                missing_cols = expected_cols - actual_cols
                extra_cols = actual_cols - expected_cols
                logging.info(f"Columns Mismatch. Missing columns: {missing_cols} | Extra columns: {extra_cols}")
        except Exception as e:
            raise CustomerChurnException(e, sys)


    def are_dtypes_valid(self, df:pd.DataFrame) -> bool:
        """
        Validates the dtypes of the data frame passed with the dtypes defined in the schema:
        """
        try:
            expected_cols = set(self.schema.keys())
            for col in expected_cols:
                expected_dtype = self.schema[col]
                actual_dtype = df[col].dtype
                if expected_dtype == actual_dtype:
                    return True
                else:
                    logging.info(f"Data-type Mismatch. Column: {col} | Expected: {expected_dtype} | Got: {actual_dtype}")
        except Exception as e:
            raise CustomerChurnException(e, sys)
        

